In [40]:
import cv2
import numpy as np
import json
from tqdm import tqdm
import polars as pl
from matplotlib import pyplot as plt
from PIL import Image

In [ ]:
with open(
    "/home/dangnh36/datasets/ecg/processed/reference_keypoints.json", "r"
) as f:
    ref_kpts = json.load(f)
ref_kpt_xys = np.array(list(ref_kpts.values()))
ref_kpt_names = list(ref_kpts.keys())
len(ref_kpt_names), ref_kpt_xys.shape

(2422, (2422, 2))

In [ ]:
pred_kpt_xys = np.load('/home/dangnh36/datasets/ecg/processed/pseudo_label/round1_finetune_dsnt/keypoints_by_round1.npy')
pred_kpt_xys.shape

(8793, 2422, 2)

In [ ]:
df = pl.read_csv('/home/dangnh36/datasets/ecg/processed/keypoints_by_round1.csv').with_row_index('index')
df

index,id,type_id,rot_code,H,keypoints
u32,i64,i64,i64,str,str
0,1449491391,1,null,null,"""{'g_0_1': array([39.37007874, …"
1,1449491391,3,null,"""[[1.0474465459666482, 0.001679…","""{'g_0_1': array([71.30957031, …"
2,1449491391,4,null,"""[[1.0457716458732285, -0.00811…","""{'g_0_1': array([62.92236328, …"
3,1449491391,5,null,"""[[0.7971640140597538, 0.067024…","""{'g_0_1': array([629.015625 ,…"
4,1449491391,6,null,"""[[0.8139604771406563, 0.009537…","""{'g_0_1': array([ 172.85154724…"
…,…,…,…,…,…
8788,3817987033,6,null,"""[[0.6940525327395047, 0.042331…","""{'g_0_1': array([424.8046875 ,…"
8789,3817987033,9,null,"""[[0.6779029018698903, -0.00180…","""{'g_0_1': array([534.515625 ,…"
8790,3817987033,10,null,"""[[0.570549926419548, -0.005112…","""{'g_0_1': array([197.859375 ,…"


In [ ]:
ori_df = pl.read_csv('/home/dangnh36/datasets/ecg/raw/train.csv')
df = df.join(ori_df, on = 'id', validate = 'm:1')
df

index,id,type_id,rot_code,H,keypoints,fs,sig_len
u32,i64,i64,i64,str,str,i64,i64
0,1449491391,1,null,null,"""{'g_0_1': array([39.37007874, …",500,5000
1,1449491391,3,null,"""[[1.0474465459666482, 0.001679…","""{'g_0_1': array([71.30957031, …",500,5000
2,1449491391,4,null,"""[[1.0457716458732285, -0.00811…","""{'g_0_1': array([62.92236328, …",500,5000
3,1449491391,5,null,"""[[0.7971640140597538, 0.067024…","""{'g_0_1': array([629.015625 ,…",500,5000
4,1449491391,6,null,"""[[0.8139604771406563, 0.009537…","""{'g_0_1': array([ 172.85154724…",500,5000
…,…,…,…,…,…,…,…
8788,3817987033,6,null,"""[[0.6940525327395047, 0.042331…","""{'g_0_1': array([424.8046875 ,…",512,5120
8789,3817987033,9,null,"""[[0.6779029018698903, -0.00180…","""{'g_0_1': array([534.515625 ,…",512,5120
8790,3817987033,10,null,"""[[0.570549926419548, -0.005112…","""{'g_0_1': array([197.859375 ,…",512,5120


In [ ]:
ref_img = cv2.imread('/home/dangnh36/datasets/ecg/raw/train/1006427285/1006427285-0001.png')
ref_img = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB)
ref_img.shape

(1700, 2200, 3)

In [46]:
print(ref_kpt_names[2365:])

['s_0_0_t', 's_0_0_b', 's_0_1_t', 's_0_1_b', 's_0_2_t', 's_0_2_b', 's_0_3_t', 's_0_3_b', 's_1_0_t', 's_1_0_b', 's_1_1_t', 's_1_1_b', 's_1_2_t', 's_1_2_b', 's_1_3_t', 's_1_3_b', 's_2_0_t', 's_2_0_b', 's_2_1_t', 's_2_1_b', 's_2_2_t', 's_2_2_b', 's_2_3_t', 's_2_3_b', 's_3_3_t', 's_3_3_b', 'dc_0_0', 'dc_0_1', 'dc_0_2', 'dc_0_3', 'dc_1_0', 'dc_1_1', 'dc_1_2', 'dc_1_3', 'dc_2_0', 'dc_2_1', 'dc_2_2', 'dc_2_3', 'dc_3_0', 'dc_3_1', 'dc_3_2', 'dc_3_3', 'n_III', 'n_aVF', 'n_V3', 'n_V6', 'n_II', 'n_aVL', 'n_V2', 'n_V5', 'n_I', 'n_aVR', 'n_V1', 'n_V4', 'n_II_full', 'mms', 'mmmv']


### Manual nearby keypoints

In [47]:
LEAD_LEFT_MID_KEYPOINTS = {
    'I': [118.11023622047246, 708.3333333333335],
    'aVR': [610.236220472441, 708.3333333333335],
    'V1': [1102.3622047244096, 708.3333333333335],
    'V4': [1594.488188976378, 708.3333333333335],

    'II_short': [118.11023622047246, 991.6666666666665],
    'aVL': [610.236220472441, 991.6666666666665],
    'V2': [1102.3622047244096, 991.6666666666665],
    'V5': [1594.488188976378, 991.6666666666665],

    'III': [118.11023622047246, 1275.0],
    'aVF': [610.236220472441, 1275.0],
    'V3': [1102.3622047244096, 1275.0],
    'V6': [1594.488188976378, 1275.0],

    'II_long': [118.11023622047246, 1534.711286089239]
}


MANUAL_LEAD_TO_NEARBY_KEYPOINT_NAMES = {
    'I': ['dc_0_0', 'dc_0_1', 'dc_0_2', 'dc_0_3', 's_0_0_t', 's_0_0_b', 'n_I', 'n_aVR'],
    'aVR': ['s_0_0_t', 's_0_0_b', 's_0_1_t', 's_0_1_b', 'n_aVR', 'n_V1'],
    'V1': ['s_0_1_t', 's_0_1_b', 's_0_2_t', 's_0_2_b', 'n_V1', 'n_V4'],
    'V4': ['s_0_2_t', 's_0_2_b', 's_0_3_t', 's_0_3_b', 'n_V4'],

    'II_short': ['dc_1_0', 'dc_1_1', 'dc_1_2', 'dc_1_3', 's_1_0_t', 's_1_0_b', 'n_II', 'n_aVL'],
    'aVL': ['s_1_0_t', 's_1_0_b', 's_1_1_t', 's_1_1_b', 'n_aVL', 'n_V2'],
    'V2': ['s_1_1_t', 's_1_1_b', 's_1_2_t', 's_1_2_b', 'n_V2', 'n_V5'],
    'V5': ['s_1_2_t', 's_1_2_b', 's_1_3_t', 's_1_3_b', 'n_V5'],

    'III': ['dc_2_0', 'dc_2_1', 'dc_2_2', 'dc_2_3', 's_2_0_t', 's_2_0_b', 'n_III', 'n_aVF'],
    'aVF': ['s_2_0_t', 's_2_0_b', 's_2_1_t', 's_2_1_b', 'n_aVF', 'n_V3'],
    'V3': ['s_2_1_t', 's_2_1_b', 's_2_2_t', 's_2_2_b', 'n_V3', 'n_V6'],
    'V6': ['s_2_2_t', 's_2_2_b', 's_2_3_t', 's_2_3_b', 'n_V6'],

    'II_long': ['dc_3_0', 'dc_3_1', 'dc_3_2', 'dc_3_3', 's_3_3_t', 's_3_3_b', 'n_II_full', 'mms', 'mmmv']
}

In [48]:
import cv2
import numpy as np

# 1. Setup Lookup Map (Name -> (x, y))
# We map all available keypoints for O(1) lookup
ref_name_to_xy = {name: xy for name, xy in zip(ref_kpt_names, ref_kpt_xys)}

# 2. Define Colors
COLOR_RED    = (255, 0, 0)    # Center
COLOR_GREEN  = (0, 255, 0)    # Neighbor
COLOR_YELLOW = (255, 255, 0)  # Connection Line
COLOR_WHITE  = (255, 255, 255)# Text

vis_img = ref_img.copy()

# Optional: Map variations like 'II_short' back to 'II' if your 
# LEAD_LEFT_MID_KEYPOINTS only uses canonical names.
lead_key_mapping = {
    'II_short': 'II',
    'II_long': 'II',
    # Add others if necessary (e.g. 'V1_long': 'V1')
}

print("--- Visualization Report ---")

for lead_name, neighbor_names in MANUAL_LEAD_TO_NEARBY_KEYPOINT_NAMES.items():
    
    # 3. Resolve the Center Coordinate
    # Try direct key first, then fallback to mapping
    if lead_name in LEAD_LEFT_MID_KEYPOINTS:
        base_key = lead_name
    elif lead_name in lead_key_mapping and lead_key_mapping[lead_name] in LEAD_LEFT_MID_KEYPOINTS:
        base_key = lead_key_mapping[lead_name]
    else:
        print(f"[SKIP] Lead center '{lead_name}' not found in LEAD_LEFT_MID_KEYPOINTS.")
        continue

    base_x, base_y = LEAD_LEFT_MID_KEYPOINTS[base_key]
    
    # Apply the offset formula to find the box center
    center_x = base_x + 1.968503937007874 * 249 / 2 if lead_name != 'II_long' else base_x + 1.968503937007874 * 999 / 2
    center_y = base_y
    center_pt = (int(center_x), int(center_y))

    # 4. Draw Lead Center
    cv2.circle(vis_img, center_pt, 6, COLOR_RED, -1, cv2.LINE_AA)
    cv2.putText(vis_img, str(lead_name), (center_pt[0]+10, center_pt[1]), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_RED, 1, cv2.LINE_AA)

    # 5. Draw Connections
    for rank, n_name in enumerate(neighbor_names):
        if n_name not in ref_name_to_xy:
            print(f"  [WARN] Neighbor '{n_name}' not found in ref_kpt_names.")
            continue

        n_xy = ref_name_to_xy[n_name]
        neighbor_pt = (int(n_xy[0]), int(n_xy[1]))

        # Draw Line
        cv2.line(vis_img, center_pt, neighbor_pt, COLOR_GREEN, 1, cv2.LINE_AA)

        # Draw Neighbor Point
        cv2.circle(vis_img, neighbor_pt, 4, COLOR_GREEN, -1, cv2.LINE_AA)

        # Draw Order Index (1-based index from the list)
        # Position text 75% of the way towards the neighbor
        t = 0.75
        text_x = int(center_pt[0] * (1 - t) + neighbor_pt[0] * t)
        text_y = int(center_pt[1] * (1 - t) + neighbor_pt[1] * t)
        
        cv2.putText(vis_img, str(rank + 1), (text_x, text_y), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, COLOR_WHITE, 1, cv2.LINE_AA)

# display(Image.fromarray(vis_img))

--- Visualization Report ---


### Auto nearby keypoints

In [49]:
import cv2
import numpy as np

# 1. Define Colors for RGB Image (R, G, B)
COLOR_RED    = (255, 0, 0)
COLOR_GREEN  = (0, 255, 0)
COLOR_YELLOW = (255, 255, 0)
COLOR_WHITE  = (255, 255, 255)

# Copy image to avoid modifying original
vis_img = ref_img.copy()

AUTO_LEAD_TO_NEARBY_KEYPOINT_NAMES = {}
TOP_K_VISUALIZE = 8
subset_ref_kpts = ref_kpt_xys[2365:]

for lead_name, (x, y) in LEAD_LEFT_MID_KEYPOINTS.items():
    # --- Calculation (Preserved) ---
    lead_center = np.array([x + 1.968503937007874 * 249 / 2 if lead_name != 'II_long' else x + 1.968503937007874 * 999 / 2 , y], dtype='float32')
    
    dists = lead_center[None] - subset_ref_kpts
    dists = (dists ** 2).sum(axis=1) ** 0.5
    
    sort_idxs = np.argsort(dists)
    nearest_names = [ref_kpt_names[2365:][idx] for idx in sort_idxs]
    AUTO_LEAD_TO_NEARBY_KEYPOINT_NAMES[lead_name] = nearest_names
    
    print(lead_name, lead_center, '-->', nearest_names[:8])

    # --- Visualization ---
    center_pt = (int(lead_center[0]), int(lead_center[1]))
    
    # 1. Draw Lead Center (Red)
    cv2.circle(vis_img, center_pt, 6, COLOR_RED, -1, cv2.LINE_AA)
    cv2.putText(vis_img, str(lead_name), (center_pt[0]+10, center_pt[1]), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_RED, 1, cv2.LINE_AA)

    # 2. Draw Connections & Order
    for rank, idx in enumerate(sort_idxs[:TOP_K_VISUALIZE]):
        neighbor_pt_arr = subset_ref_kpts[idx]
        neighbor_pt = (int(neighbor_pt_arr[0]), int(neighbor_pt_arr[1]))
        
        # Draw Line (Yellow)
        cv2.line(vis_img, center_pt, neighbor_pt, COLOR_GREEN, 1, cv2.LINE_AA)
        
        # Draw Neighbor Point (Green)
        cv2.circle(vis_img, neighbor_pt, 4, COLOR_GREEN, -1, cv2.LINE_AA)
        
        # Draw Order Index (1, 2, 3...)
        # We calculate a position 80% along the line towards the neighbor
        # to prevent text clustering at the center but keep it near the target.
        t = 0.8 
        text_x = int(center_pt[0] * (1 - t) + neighbor_pt[0] * t)
        text_y = int(center_pt[1] * (1 - t) + neighbor_pt[1] * t)
        
        cv2.putText(vis_img, str(rank + 1), (text_x, text_y), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_WHITE, 1, cv2.LINE_AA)

# Note: If saving with cv2.imwrite later, remember to convert RGB -> BGR
# cv2.imwrite("output.jpg", cv2.cvtColor(vis_img, cv2.COLOR_RGB2BGR))

# display(Image.fromarray(vis_img))
print(AUTO_LEAD_TO_NEARBY_KEYPOINT_NAMES)

I [363.18896 708.3333 ] --> ['n_I', 'dc_0_2', 's_0_0_t', 's_0_0_b', 'dc_0_1', 'n_aVR', 'dc_0_3', 'dc_0_0']
aVR [855.31494 708.3333 ] --> ['n_aVR', 's_0_0_t', 's_0_0_b', 's_0_1_t', 's_0_1_b', 'n_V1', 's_1_0_t', 's_1_1_t']
V1 [1347.4409  708.3333] --> ['n_V1', 's_0_1_t', 's_0_1_b', 's_0_2_t', 's_0_2_b', 'n_V4', 's_1_1_t', 's_1_2_t']
V4 [1839.5669  708.3333] --> ['n_V4', 's_0_3_t', 's_0_3_b', 's_0_2_t', 's_0_2_b', 's_1_3_t', 's_1_2_t', 's_1_3_b']
II_short [363.18896 991.6667 ] --> ['n_II', 'dc_1_2', 's_1_0_b', 's_1_0_t', 'dc_1_1', 'n_aVL', 'dc_1_3', 'dc_1_0']
aVL [855.31494 991.6667 ] --> ['n_aVL', 's_1_0_b', 's_1_0_t', 's_1_1_b', 's_1_1_t', 'n_V2', 'n_aVR', 's_2_0_t']
V2 [1347.4409  991.6667] --> ['n_V2', 's_1_1_b', 's_1_1_t', 's_1_2_b', 's_1_2_t', 'n_V5', 'n_V1', 's_2_1_t']
V5 [1839.5669  991.6667] --> ['n_V5', 's_1_3_b', 's_1_3_t', 's_1_2_b', 's_1_2_t', 'n_V4', 's_2_3_t', 's_0_3_b']
III [ 363.18896 1275.     ] --> ['n_III', 'dc_2_2', 's_2_0_b', 's_2_0_t', 'dc_2_1', 'n_aVF', 'dc_2_3', '

In [50]:
1700 / 21.59 * 10

787.4015748031496

In [51]:
1042.204724409449 / 512

2.035556102362205

In [52]:
250 * 2.5

625.0

In [53]:
492.12598425196853 / (500 / 512)

503.93700787401576

In [54]:
1700 / 21.59 * (500 / 492.12598425196853) * 1

80.0

In [55]:
import cv2
import numpy as np
import torch
from PIL import Image
from IPython.display import display

def show_visualization_grid(base_img, heatmap, layout='1x3'):
    """
    Displays a grid of [Base Image, Colored Heatmap, Overlay].
    
    Args:
        base_img: Numpy array (H, W) or (H, W, 3)
        heatmap: Numpy array or Torch tensor (H, W)
        layout: '1x3' (horizontal) or '3x1' (vertical)
    """
    # 1. Prepare Heatmap (Tensor -> Numpy -> Norm -> Color)
    if isinstance(heatmap, torch.Tensor):
        hm_data = heatmap.detach().cpu().numpy()
    else:
        hm_data = heatmap
        
    # Normalize to 0-255
    hm_norm = (hm_data - hm_data.min()) / (hm_data.max() - hm_data.min() + 1e-8)
    hm_uint8 = (hm_norm * 255).astype(np.uint8)
    
    # Apply Jet Colormap (Blue=Low, Red=High) & Convert to RGB
    hm_color = cv2.applyColorMap(hm_uint8, cv2.COLORMAP_JET)
    hm_color = cv2.cvtColor(hm_color, cv2.COLOR_BGR2RGB)

    # 2. Prepare Base Image (Ensure RGB)
    img_rgb = base_img.copy()
    if len(img_rgb.shape) == 2:
        img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_GRAY2RGB)

    # 3. Resize Heatmap to match Base Image (for overlay/grid)
    target_h, target_w = img_rgb.shape[:2]
    if hm_color.shape[:2] != (target_h, target_w):
        hm_color = cv2.resize(hm_color, (target_w, target_h), interpolation=cv2.INTER_LINEAR)

    # 4. Create Overlay
    alpha, beta = 0.6, 0.4
    overlay = cv2.addWeighted(img_rgb, alpha, hm_color, beta, 0)

    # 5. Arrange Grid
    # Add simple borders/labels logic if needed, but here we just concat
    images = [img_rgb, hm_color, overlay]
    
    if layout == '1x3':
        grid = np.hstack(images)
    elif layout == '3x1':
        grid = np.vstack(images)
    else:
        raise ValueError("Layout must be '1x3' or '3x1'")

    print(f"Displaying grid ({layout}): {grid.shape}")
    display(Image.fromarray(grid))

In [ ]:
import os
import torch
import cv2
import numpy as np
import cv2
import numpy as np
from scipy.interpolate import RectBivariateSpline

# signal range
# GLOBAL MIN: -6.136
# GLOBAL_MAX: 7.1

# GLOBAL PLOT X/Y MIN/MAX: 0 11.176 0 21.59

REF_KPT_NAMES = ref_kpt_names
REF_KPT_XYS = ref_kpt_xys
MANUAL_LEAD_TO_NEARBY_KEYPOINT_INDICES = {
    lead_name: [REF_KPT_NAMES.index(kpt_name) for kpt_name in nearby_kpt_names] for lead_name, nearby_kpt_names in MANUAL_LEAD_TO_NEARBY_KEYPOINT_NAMES.items()
}
print('AUTO LEAD TO NEARBY KEYPOINT INDICES:', MANUAL_LEAD_TO_NEARBY_KEYPOINT_INDICES, sep = '\n')


def compute_crop_left_top(x_left, y_mid, signal_length_secs, crop_h, crop_w = 512):
    # number of signal steps: secs * fs
    # step = 2200 / 11.176 * (1 / fs)
    # signal_pixels = step * secs * fs = 2200 / 11.176 * (1 / fs) * secs * fs
    signal_pixels = 2200 / 11.176 * signal_length_secs
    pad_left = (crop_w - signal_pixels) / 2
    crop_left = x_left - pad_left
    crop_top = y_mid - crop_h / 2
    return crop_left, crop_top
    

def crop_by_warp(lead_name, img, detected_keypoints, signal_length_secs = 2.5,
                 warp_h = 512, warp_w = 512, crop_h = 512, crop_w = 512,
                 filter_nearby = True,
                 nearby_keypoint_indices = MANUAL_LEAD_TO_NEARBY_KEYPOINT_INDICES):
    if filter_nearby:
        nearby_kpt_indices = nearby_keypoint_indices[lead_name]
    else:
        nearby_kpt_indices = list(range(len(REF_KPT_XYS)))
    # reference keypoints in REFERENCE IMAGE 0001
    ref_kpt_xys = REF_KPT_XYS[nearby_kpt_indices]
    # detected keypoints in current image
    detected_kpt_xys = detected_keypoints[nearby_kpt_indices]
    print(ref_kpt_xys.shape, detected_kpt_xys.shape)
    
    ref_x_left, ref_y_mid = LEAD_LEFT_MID_KEYPOINTS[lead_name]
    crop_left, crop_top = compute_crop_left_top(ref_x_left, ref_y_mid, signal_length_secs, crop_h, crop_w)
    print('CROP RECT:', crop_left, crop_top, crop_h, crop_w)
    # now, left_top has contigous coordinate of (0, 0)
    inside_crop_ref_kpt_xys = ref_kpt_xys - np.array([[crop_left, crop_top]])
    # scale
    inside_warp_ref_kpt_xys = inside_crop_ref_kpt_xys * np.array([[warp_w / crop_w, warp_h / crop_h]], dtype = 'float32')

    # @TODO - for loop with increasing reproj_threshold
    # find homo transformation matrix from current image to warped image
    # -0.5 to convert from contigous physic coordiate to pixel indices
    H, mask = cv2.findHomography(
            detected_kpt_xys - 0.5,
            inside_warp_ref_kpt_xys - 0.5,
            method=cv2.USAC_MAGSAC,
            ransacReprojThreshold=1,
            confidence=0.9999,
            maxIters=10_000,
    )
    print(f'match: {mask.sum()} / {len(mask)}, H={H.shape}')

    warp_img = cv2.warpPerspective(img, H, (warp_w, warp_h), flags = cv2.INTER_CUBIC, borderMode = cv2.BORDER_CONSTANT, borderValue = 0)
    return warp_img


import cv2
import numpy as np
import torch
import torch.nn.functional as F
from scipy.ndimage import map_coordinates

def apply_backend_remap(img, map_x, map_y, backend='cv2', device='cpu'):
    """
    Applies the remapping (warping) using the specified backend.
    
    Args:
        img: (H, W, C) or (H, W) numpy array.
        map_x: (H_out, W_out) float32 array of absolute X pixel coordinates.
        map_y: (H_out, W_out) float32 array of absolute Y pixel coordinates.
        backend: 'cv2', 'scipy', or 'torch'.
        device: 'cpu' or 'cuda' (only used if backend='torch').
    """
    # Ensure maps are float32
    map_x = map_x.astype(np.float32)
    map_y = map_y.astype(np.float32)
    
    # ---------------------------------------------------------
    # 1. OpenCV Backend (Fastest CPU)
    # ---------------------------------------------------------
    if backend == 'cv2':
        # -0.5 to convert from contigous coordinate space to pixel indices
        return cv2.remap(img, map_x - 0.5, map_y - 0.5, interpolation=cv2.INTER_LANCZOS4)

    # ---------------------------------------------------------
    # 2. SciPy Backend (Precise, Scientific)
    # ---------------------------------------------------------
    elif backend == 'scipy':
        # scipy.ndimage.map_coordinates expects coordinates in (Row, Col) order, i.e., (y, x).
        # It also expects a shape of (2, N_points), in contigous coordinate space
        # so we don't need  -0.5, keep contigous as is
        coords = np.stack([map_y, map_x])
        
        # Handle 2D (Grayscale) vs 3D (Color)
        if img.ndim == 2:
            # order=3 is Bicubic, mode='nearest' matches cv2 border logic
            return map_coordinates(img, coords, order=3, mode='nearest')
        elif img.ndim == 3:
            # Loop over channels (R, G, B)
            channels = []
            for c in range(img.shape[2]):
                channels.append(map_coordinates(img[..., c], coords, order=3, mode='nearest'))
            return np.stack(channels, axis=-1)

    # ---------------------------------------------------------
    # 3. PyTorch Backend (GPU Accelerated, Differentiable)
    # ---------------------------------------------------------
    elif backend == 'torch':
        H, W = img.shape[:2]
        
        # Convert Image to Tensor: (H, W, C) -> (1, C, H, W)
        if img.ndim == 2:
            img_tensor = torch.from_numpy(img)[None, None, ...].float()
        else:
            # Transpose to Channel-First
            img_tensor = torch.from_numpy(img.transpose(2, 0, 1))[None, ...].float()
            
        img_tensor = img_tensor.to(device)

        # Normalize Coordinates to [-1, 1] for grid_sample
        # We use align_corners=False convention
        norm_x = 2.0 * map_x / W - 1.0
        norm_y = 2.0 * map_y / H - 1.0
        
        # Stack to (1, H_out, W_out, 2)
        grid = np.stack([norm_x, norm_y], axis=-1)
        grid_tensor = torch.from_numpy(grid)[None, ...].float().to(device)
        
        # Sampling
        out_tensor = F.grid_sample(img_tensor, grid_tensor, mode='bicubic', padding_mode='border', align_corners=False)
        
        # Convert back to Numpy: (1, C, H, W) -> (H, W, C)
        out_np = out_tensor.detach().cpu().numpy()[0]
        if img.ndim == 3:
            out_np = out_np.transpose(1, 2, 0)
        else:
            out_np = out_np[0] # Squeeze channel dim for grayscale
            
        return np.clip(out_np, 0, 255).astype(img.dtype)

    else:
        raise ValueError(f"Unknown backend: {backend}")


from scipy.interpolate import griddata

def unwrap_with_griddata(img, detected_grid, reference_grid, warp_grid, 
                         method='linear', map_smooth_sigma=0.0, 
                         backend='cv2', device='cpu'):
    # ... (Same setup code as before) ...
    points = reference_grid.reshape(-1, 2)
    values_x = detected_grid[:, :, 0].flatten()
    values_y = detected_grid[:, :, 1].flatten()
    xi = warp_grid.reshape(-1, 2)

    # Interpolation
    print(f"Interpolating flow map using griddata ({method})...")
    map_x_flat = griddata(points, values_x, xi, method=method)
    map_y_flat = griddata(points, values_y, xi, method=method)

    # Handle NaNs
    if np.any(np.isnan(map_x_flat)) or np.any(np.isnan(map_y_flat)):
        nan_mask = np.isnan(map_x_flat)
        map_x_flat[nan_mask] = griddata(points, values_x, xi[nan_mask], method='nearest')
        map_y_flat[nan_mask] = griddata(points, values_y, xi[nan_mask], method='nearest')

    out_h, out_w = warp_grid.shape[:2]
    map_x = map_x_flat.reshape(out_h, out_w).astype(np.float32)
    map_y = map_y_flat.reshape(out_h, out_w).astype(np.float32)

    # Smoothing
    if map_smooth_sigma > 0:
        ksize = int(2 * round(3 * map_smooth_sigma) + 1)
        if ksize < 3: ksize = 3
        map_x = cv2.GaussianBlur(map_x, (ksize, ksize), map_smooth_sigma)
        map_y = cv2.GaussianBlur(map_y, (ksize, ksize), map_smooth_sigma)

    # --- CHANGED: Unified Backend Call ---
    print(f"Remapping ({backend})...")
    return apply_backend_remap(img, map_x, map_y, backend=backend, device=device)


from scipy.interpolate import RectBivariateSpline

def unwrap_with_rect_spline(img, detected_grid, reference_grid, warp_grid, 
                            map_smooth_sigma=0.0, backend='cv2', device='cpu'):
    # ... (Same setup code as before) ...
    grid_rows = reference_grid[:, 0, 1] 
    grid_cols = reference_grid[0, :, 0]

    if not (np.all(np.diff(grid_rows) > 0) and np.all(np.diff(grid_cols) > 0)):
        raise ValueError("Reference grid axes must be strictly increasing.")

    src_x_vals = detected_grid[:, :, 0]
    src_y_vals = detected_grid[:, :, 1]

    spline_x = RectBivariateSpline(grid_rows, grid_cols, src_x_vals, kx=3, ky=3, s=0)
    spline_y = RectBivariateSpline(grid_rows, grid_cols, src_y_vals, kx=3, ky=3, s=0)

    query_x = warp_grid[:, :, 0].flatten()
    query_y = warp_grid[:, :, 1].flatten()

    map_x_flat = spline_x(query_y, query_x, grid=False)
    map_y_flat = spline_y(query_y, query_x, grid=False)

    out_h, out_w = warp_grid.shape[:2]
    map_x = map_x_flat.reshape(out_h, out_w).astype(np.float32)
    map_y = map_y_flat.reshape(out_h, out_w).astype(np.float32)

    if map_smooth_sigma > 0:
        ksize = int(2 * round(3 * map_smooth_sigma) + 1)
        if ksize < 3: ksize = 3
        map_x = cv2.GaussianBlur(map_x, (ksize, ksize), map_smooth_sigma)
        map_y = cv2.GaussianBlur(map_y, (ksize, ksize), map_smooth_sigma)

    # --- CHANGED: Unified Backend Call ---
    return apply_backend_remap(img, map_x, map_y, backend=backend, device=device)


def unwrap_with_piecewise_homography(img, detected_grid, reference_grid, warp_grid, 
                                     K=4, map_smooth_sigma=0.0, 
                                     backend='cv2', device='cpu'):
    # ... (Same setup code as before) ...
    grid_rows = reference_grid[:, 0, 1]
    grid_cols = reference_grid[0, :, 0]
    rows = len(grid_rows)
    cols = len(grid_cols)

    H_matrices = np.zeros((rows - 1, cols - 1, 3, 3), dtype=np.float32)
    search_radius = int(np.sqrt(K)) + 2

    for r in range(rows - 1):
        for c in range(cols - 1):
            if K == 4:
                # Fast path code (same as previous)
                dst_pts = np.array([detected_grid[r, c], detected_grid[r, c+1], detected_grid[r+1, c+1], detected_grid[r+1, c]], dtype=np.float32)
                src_pts = np.array([[grid_cols[c], grid_rows[r]], [grid_cols[c+1], grid_rows[r]], [grid_cols[c+1], grid_rows[r+1]], [grid_cols[c], grid_rows[r+1]]], dtype=np.float32)
                H_matrices[r, c] = cv2.getPerspectiveTransform(src_pts, dst_pts)
            else:
                # --- ROBUST PATH: K Nearest Neighbors ---
                center_r, center_c = r + 0.5, c + 0.5
                r_min = max(0, int(center_r - search_radius))
                r_max = min(rows, int(center_r + search_radius + 1))
                c_min = max(0, int(center_c - search_radius))
                c_max = min(cols, int(center_c + search_radius + 1))
                
                rr, cc = np.meshgrid(np.arange(r_min, r_max), np.arange(c_min, c_max), indexing='ij')
                rr_flat, cc_flat = rr.flatten(), cc.flatten()
                
                dists_sq = (rr_flat - center_r)**2 + (cc_flat - center_c)**2
                k_indices = np.argsort(dists_sq)[:K]
                
                sel_r, sel_c = rr_flat[k_indices], cc_flat[k_indices]
                
                src_pts = reference_grid[sel_r, sel_c].astype(np.float32)
                dst_pts = detected_grid[sel_r, sel_c].astype(np.float32)
                
                H, _ = cv2.findHomography(src_pts, dst_pts, method=0)
                if H is None:
                    # Fallback to K=4 logic
                    dst_bk = np.array([detected_grid[r,c], detected_grid[r,c+1], detected_grid[r+1,c+1], detected_grid[r+1,c]], dtype=np.float32)
                    src_bk = np.array([[grid_cols[c], grid_rows[r]], [grid_cols[c+1], grid_rows[r]], [grid_cols[c+1], grid_rows[r+1]], [grid_cols[c], grid_rows[r+1]]], dtype=np.float32)
                    H = cv2.getPerspectiveTransform(src_bk, dst_bk)
                
                H_matrices[r, c] = H

    x_coords = warp_grid[:, :, 0]
    y_coords = warp_grid[:, :, 1]
    
    col_indices = np.searchsorted(grid_cols, x_coords, side='right') - 1
    row_indices = np.searchsorted(grid_rows, y_coords, side='right') - 1
    col_indices = np.clip(col_indices, 0, cols - 2)
    row_indices = np.clip(row_indices, 0, rows - 2)
    print(row_indices, col_indices, sep='\n')

    print('H mat:', H_matrices.shape)

    pixel_H = H_matrices[row_indices, col_indices]
    
    ones = np.ones_like(x_coords)
    points_homogeneous = np.stack([x_coords, y_coords, ones], axis=-1)[..., np.newaxis]
    mapped_homogeneous = np.matmul(pixel_H, points_homogeneous)
    
    w_vec = mapped_homogeneous[:, :, 2, 0]
    w_vec[w_vec == 0] = 1e-8
    
    map_x = (mapped_homogeneous[:, :, 0, 0] / w_vec).astype(np.float32)
    map_y = (mapped_homogeneous[:, :, 1, 0] / w_vec).astype(np.float32)

    if map_smooth_sigma > 0:
        ksize = int(2 * round(3 * map_smooth_sigma) + 1)
        if ksize < 3: ksize = 3
        map_x = cv2.GaussianBlur(map_x, (ksize, ksize), map_smooth_sigma)
        map_y = cv2.GaussianBlur(map_y, (ksize, ksize), map_smooth_sigma)

    # # visualize
    # print("Flow Map Shapes:", map_x.shape)
    
    # # Calculate Magnitude
    # # Note: Since map_x/y are absolute coordinates, this shows the distance from (0,0).
    # # If you wanted to see the *distortion*, you would subtract the identity grid first.
    # _map_xy = np.sqrt(map_x ** 2 + map_y ** 2)
    
    # maps_to_show = [('Row Index', row_indices), ('Col Index', col_indices), ('Map X', map_x), ('Map Y', map_y), ('Magnitude', _map_xy)]
    
    # for name, _map in maps_to_show:
    #     # 1. Normalize to 0.0 - 1.0
    #     _min, _max = _map.min(), _map.max()
    #     if _max - _min > 1e-8:
    #         _norm = (_map - _min) / (_max - _min)
    #     else:
    #         _norm = np.zeros_like(_map)
    
    #     # 2. Convert to uint8 (0-255)
    #     _uint8 = (_norm * 255).astype('uint8')
    
    #     # 3. Apply ColorMap (Blue=Low, Red=High)
    #     # cv2.applyColorMap returns a BGR image
    #     _color_bgr = cv2.applyColorMap(_uint8, cv2.COLORMAP_JET)
    
    #     # 4. Convert BGR to RGB (for PIL/Display)
    #     _color_rgb = cv2.cvtColor(_color_bgr, cv2.COLOR_BGR2RGB)
    
    #     print(f"Visualizing: {name} (Range: {_min:.2f} to {_max:.2f})")
    #     display(Image.fromarray(_color_rgb))
    

    # --- CHANGED: Unified Backend Call ---
    return apply_backend_remap(img, map_x, map_y, backend=backend, device=device)



def crop_by_piecewise_homography(lead_name, img, detected_keypoints, signal_length_secs = 2.5,
                 warp_h = 512, warp_w = 512, crop_h = 512, crop_w = 512, filter_inside = True):

    ref_x_left, ref_y_mid = LEAD_LEFT_MID_KEYPOINTS[lead_name]
    crop_left, crop_top = compute_crop_left_top(ref_x_left, ref_y_mid, signal_length_secs, crop_h, crop_w)
    print('CROP RECT:', crop_left, crop_top, crop_h, crop_w)

    candidate_ref_kpt_xys = REF_KPT_XYS[:2365]
    if filter_inside:
        is_kpt_inside_crop_indices = np.nonzero(np.all((candidate_ref_kpt_xys >= np.array([[crop_left, crop_top]])) & 
                                             (candidate_ref_kpt_xys <= np.array([[crop_left + crop_w, crop_top + crop_h]])), axis = 1))[0]
        # inside_crop_names = [REF_KPT_NAMES[idx] for idx in is_ref_kpt_inside_crop_indices]
        ref_kpt_idxs = np.arange(0, 43 * 55).reshape(43, 55)
        
        # ref_kpt_row_idxs = np.arange(0, 43)[:, None].repeat(55, 1).flatten()
        # ref_kpt_col_idxs = np.arange(0, 55)[None].repeat(43, 0).flatten()
        # inside_crop_row_idxs = ref_kpt_row_idxs[is_ref_kpt_inside_crop_indices]
        # inside_crop_col_idxs = ref_kpt_col_idxs[is_ref_kpt_inside_crop_indices]
        # inside_num_rows = inside_crop_row_idxs.max() - inside_crop_row_idxs.min() + 1
        # inside_num_cols = inside_crop_col_idxs.max() - inside_crop_col_idxs.min() + 1
    
        inside_kpt_idxs = ref_kpt_idxs.flatten()[is_kpt_inside_crop_indices]
        inside_kpt_idx_min = inside_kpt_idxs.min()
        inside_kpt_idx_max = inside_kpt_idxs.max()
        
        _row_start = max(0, inside_kpt_idx_min // 55 - 1)
        _row_end = min(inside_kpt_idx_max // 55 + 2, 43)
        _row_num = _row_end - _row_start
        _col_start = max(0, inside_kpt_idx_min % 55 - 1)
        _col_end = min(inside_kpt_idx_max % 55 + 2, 55)
        _col_num = _col_end - _col_start
        
        # extend 1 cell along all 4 sides
        inside_kpt_idxs2 = ref_kpt_idxs[_row_start: _row_end, _col_start: _col_end].flatten()
        inside_crop_ref_xys = candidate_ref_kpt_xys[inside_kpt_idxs2].reshape(_row_num, _col_num, 2)
        inside_detected_keypoints = detected_keypoints[inside_kpt_idxs2].reshape(_row_num, _col_num, 2)
    else:
        inside_crop_ref_xys = candidate_ref_kpt_xys.reshape(43, 55, 2)
        inside_detected_keypoints = detected_keypoints[:2365].reshape(43, 55, 2)

    # create the expected output reference grid points
    warp_ref_grid_x = np.linspace(crop_left + 0.5, crop_left + crop_w - 0.5, warp_w)
    warp_ref_grid_y = np.linspace(crop_top + 0.5, crop_top + crop_h - 0.5, warp_h)
    warp_ref_grid_xys = np.stack(np.meshgrid(warp_ref_grid_x, warp_ref_grid_y, indexing = "xy"), axis = -1)
    
    # warp_img = unwrap_with_rect_spline(img, inside_detected_keypoints, inside_crop_ref_xys, warp_ref_grid_xys, map_smooth_sigma=11)
    warp_img = unwrap_with_piecewise_homography(img, inside_detected_keypoints, inside_crop_ref_xys, warp_ref_grid_xys, K=4, map_smooth_sigma=0.0)
    # warp_img = unwrap_with_griddata(img, inside_detected_keypoints, inside_crop_ref_xys, warp_ref_grid_xys, method='linear', map_smooth_sigma=0.0)
    return warp_img
    

def resize_1d_cv2(arr, target_len):
    """
    Resizes a 1D array.
    
    - Upsampling: Uses Cubic interpolation (smoothness).
    - Downsampling: Uses Area interpolation (preserves sum/intensity).
    """
    # Ensure input is a numpy array
    arr = np.array(arr)
    input_len = len(arr)
    
    # Bypass if no change
    if input_len == target_len:
        return arr

    # Reshape to (1, W) because cv2.resize expects an image (2D)
    # We treat the 1D array as a 1-pixel tall image.
    img_1d = arr.reshape(1, -1)

    # Choose interpolation method based on resizing direction
    if target_len < input_len:
        # DOWN-sampling: Use INTER_AREA.
        # This averages pixel values, ensuring peaks aren't "skipped".
        method = cv2.INTER_AREA
    else:
        # UP-sampling: Use INTER_LINEAR (or INTER_CUBIC).
        # This creates smooth transitions between points.
        method = cv2.INTER_CUBIC

    # Resize
    # Note: cv2.resize size argument is (Width, Height) -> (target_len, 1)
    result = cv2.resize(img_1d, (target_len, 1), interpolation=method)
    
    # Flatten back to 1D
    return result.flatten()


class PerColumnHeatmapCodec:
    def __init__(self, sigma=2, crop_aspect_ratio = 1.0, H = 512, W = 512, L = 500,
                 dtype = torch.float32, cutoff_thres = None, subpixel = False, adaptive_sigma_scale=None):
        # cutoff_thres follow 3-sigma rule for 1D normal distribution: 0.011108996538242308
        # re-compute sigma**2 (var) relative to gt image (from relative to reference image)
        self.gt_sigma = sigma * (L / 492.12598425196853) / crop_aspect_ratio
        self.crop_aspect_ratio = crop_aspect_ratio
        self.H = H
        self.W = W
        self.L = L
        self.dtype = dtype
        self.cutoff_thres = cutoff_thres
        self.subpixel = subpixel
        self.adaptive_sigma_scale = adaptive_sigma_scale
        self.pad_left = (W - L) // 2
        self.pad_right = W - L - self.pad_left
        assert self.pad_left == self.pad_right
        self.signal_scale_factor = 1700 / 21.59 * (L / 492.12598425196853) / crop_aspect_ratio
        # (H, L)
        self.grid = torch.linspace(0.5, H - 0.5, H, dtype = dtype).unsqueeze(1).expand(-1, L)

    def encode(self, signal):
        # proper interpolate signal to gt_length
        resized_signal = resize_1d_cv2(signal, self.L)
    
        # scale the signal
        scaled_signal = -resized_signal * self.signal_scale_factor + self.H / 2
        scaled_signal = torch.from_numpy(scaled_signal).to(self.dtype)

        # compute adaptive sigma
        if self.adaptive_sigma_scale is not None:
            arr = scaled_signal
            local_abs_diff = (np.abs(arr - np.r_[arr[0], arr[:-1]]) + np.abs(arr - np.r_[arr[1:], arr[-1]])) / 2
            # 3-sigma rule: if > 3*sigma, start using scale >= 1
            # _large_mask = local_abs_diff > 3 * self.gt_sigma
            # print('DEBUG:', _large_mask.sum(), len(_large_mask), 3 * self.gt_sigma)
            gt_sigma = self.gt_sigma + self.adaptive_sigma_scale * np.maximum(local_abs_diff - 3 * self.gt_sigma, 0) / 3
            # print('SIGMA:', gt_sigma.min(), gt_sigma.max())
        else:
            gt_sigma = self.gt_sigma
            
        heatmap = torch.zeros((self.H, self.W), dtype = self.dtype)
        # generate gaussian-like heatmap
        # (H, L) - (1, L)
        roi_heatmap = torch.exp(-0.5 * (self.grid - scaled_signal.unsqueeze(0)) ** 2 / gt_sigma ** 2)
        if self.cutoff_thres is not None:
            roi_heatmap[roi_heatmap < self.cutoff_thres] = 0.0

        # refine to obtain sub-pixel accuracy
        if self.subpixel:
            cur_expectation = torch.sum(self.grid * roi_heatmap, axis = 0) / torch.sum(roi_heatmap, axis = 0)
            diff = cur_expectation - scaled_signal
            # negative vs positive can cause under/over flow when trying to mul/add
            # not to be easy, take it later
            raise NotImplementedError
            
        heatmap[:, self.pad_left:-self.pad_right] = roi_heatmap
        return heatmap

    def decode(self, output):
        raise NotImplementedError





# SAMPLE_IDX = 59
SAMPLE_IDX = 9 * 1 + 6
sample = df[SAMPLE_IDX].to_dicts()[0]
sample_id = sample['id']
type_id = sample['type_id']
rot_code = sample['rot_code']
fs = sample['fs']
sig_len = sample['sig_len']
img_path = os.path.join('/home/dangnh36/datasets/ecg/raw/train/', str(sample_id), f"{sample_id}-{type_id:04d}.png")
signal_csv_path = os.path.join('/home/dangnh36/datasets/ecg/raw/train/', str(sample_id), f'{sample_id}.csv')
signal_df = pl.scan_csv(signal_csv_path).select(pl.col('*').cast(pl.Float64)).collect()
display(signal_df)
img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
# correct the orientation
if rot_code is not None:
    assert rot_code == int(rot_code)
    rot_code = int(rot_code)
    img = cv2.rotate(img, rot_code)
    
print(img.shape)

detected_keypoints = pred_kpt_xys[SAMPLE_IDX]


HEATMAP_STRIDE = 1
GT_H, GT_W, GT_L = 512, 512, 500
# GT_H, GT_W, GT_L = 1024, 1024, 1000
WARP_H, WARP_W = round(GT_H * HEATMAP_STRIDE), round(GT_W * HEATMAP_STRIDE)

SIGNAL_LENGTH_SECS = 2.5
KEEP_ASPECT_RATIO = False
# signal cover exactly 500 pixels width in groundtruth mask (width=512 pixels)
# 503.93700787401576 = 492.12598425196853 / (500 / 512)
CROP_W = 492.12598425196853 / (GT_L / GT_W)
CROP_H = CROP_W * 1
CUTOFF_THRES =  0.011108996538242308
SIGMA = 2
ADAPTIVE_SIGMA_SCALE = 0.5

if KEEP_ASPECT_RATIO:
    _CROP_H = CROP_W * WARP_H / WARP_W
    if CROP_H is None:
        CROP_H = _CROP_H
    else:
        assert CROP_H == _CROP_H
else:
    assert CROP_H is not None


codec = PerColumnHeatmapCodec(
    sigma=SIGMA, crop_aspect_ratio = CROP_H / CROP_W, H = GT_H, W = GT_W, L = GT_L,
    cutoff_thres = CUTOFF_THRES, dtype = torch.float32, adaptive_sigma_scale=ADAPTIVE_SIGMA_SCALE
)


for lead_name in MANUAL_LEAD_TO_NEARBY_KEYPOINT_NAMES.keys():
    if lead_name == 'II_long':
        continue

    if lead_name != 'V4':
        continue

        
    print(f'--------------- {lead_name} ----------------')
    
    # warp_img = crop_by_warp(
    #     lead_name, img, detected_keypoints, signal_length_secs = SIGNAL_LENGTH_SECS,
    #     warp_h = WARP_H, warp_w = WARP_W, crop_h = CROP_H, crop_w = CROP_W, filter_nearby = True
    # )
    
    warp_img = crop_by_piecewise_homography(
        lead_name, img, detected_keypoints, signal_length_secs = SIGNAL_LENGTH_SECS,
        warp_h = WARP_H, warp_w = WARP_W, crop_h = CROP_H, crop_w = CROP_W, filter_inside = True
    )
    
    print("WARP IMAGE:", warp_img.shape)
    
    # generate GT
    if lead_name == 'II_short':
        _lead_name = 'II'
    else:
        _lead_name = lead_name
    lead_signal = signal_df[_lead_name].to_numpy()
    lead_signal = lead_signal[~np.isnan(lead_signal)].astype('float32')
    if lead_name == 'II_short':
        lead_signal = lead_signal[:int(len(lead_signal) / 4)]
    print('SIGNAL:', lead_signal.shape, sig_len, fs)

    gt_heatmap = codec.encode(lead_signal)
    print('GT HEATMAP:', gt_heatmap.shape, gt_heatmap.min(), gt_heatmap.max())

    # visualize
    show_visualization_grid(warp_img, gt_heatmap, layout='1x3')

In [ ]:
print(REF_KPT_XYS.tolist())